In [5]:
from __future__ import annotations

import os
import re
import shutil
from math import ceil
from pathlib import Path
from typing import List, Dict, Any

import fitz  # PyMuPDF
import pdfplumber
import torch
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

BASE_DIR = Path('C:/skn24/allianz-rag')

DATA_DIR = BASE_DIR / "data" / "raw"
DB_DIR = BASE_DIR / "vectordb"
COLLECTION_NAME = "allianz_care"

ENV_PATH = BASE_DIR / ".env"
load_dotenv(dotenv_path=ENV_PATH)

True

In [ ]:

# =========================
# 실행 옵션
# =========================
EMBED_MODEL_NAME = os.getenv("EMBED_MODEL_NAME", "BAAI/bge-m3")
EMBED_DEVICE = os.getenv("EMBED_DEVICE", "cpu")  # GPU 있으면 cuda
EMBED_BATCH_SIZE = int(os.getenv("EMBED_BATCH_SIZE", "16"))
INDEX_BATCH_SIZE = int(os.getenv("INDEX_BATCH_SIZE", "100"))
RESET_VECTORDB = os.getenv("RESET_VECTORDB", "true").lower() == "true"

TORCH_NUM_THREADS = int(
    os.getenv("TORCH_NUM_THREADS", str(max(1, (os.cpu_count() or 4) - 1)))
)

torch.set_num_threads(TORCH_NUM_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass


# 1. 파일 목록
FILES: List[Dict[str, Any]] = [
    # 글로벌 공통
    {
        "path": DATA_DIR / "DOC-Care-IBG-EN-1125_개인고객용혜택가이드.pdf",
        "doc_type": "benefit_guide",
        "doc_year": 2025,
        "region": "global",
        "product_family": "care_global",
    },
    {
        "path": DATA_DIR / "care-tob-en_보장금액.pdf",
        "doc_type": "tob",
        "doc_year": 2025,
        "region": "global",
        "product_family": "care_global",
    },
    {
        "path": DATA_DIR / "FRM-PreAuth-EN-0825_사전승인신청서.pdf",
        "doc_type": "preauth_form",
        "doc_year": 2025,
        "region": "global",
        "product_family": "care_global",
    },
    {
        "path": DATA_DIR / "FRM-PCF-EN-1125_사후보험청구서.pdf",
        "doc_type": "claim_form",
        "doc_year": 2025,
        "region": "global",
        "product_family": "care_global",
    },

    # 지역별 코퍼스
    {
        "path": DATA_DIR / "DOC-Singapore-IBG-EN-0126_싱가포르.pdf",
        "doc_type": "benefit_guide",
        "doc_year": 2026,
        "region": "singapore",
        "product_family": "regional",
    },
    {
        "path": DATA_DIR / "DOC-IBG-Dubai-Northern-Emirates-EN-0126_두바이.pdf",
        "doc_type": "benefit_guide",
        "doc_year": 2026,
        "region": "dubai_northern_emirates",
        "product_family": "regional",
    },
    {
        "path": DATA_DIR / "DOC-LEBANON-IBG-EN-0725_레바논.pdf",
        "doc_type": "benefit_guide",
        "doc_year": 2025,
        "region": "lebanon",
        "product_family": "regional",
    },
    {
        "path": DATA_DIR / "DOC-IBG-Indonesia-en-UK-1123_인도네시아.pdf",
        "doc_type": "benefit_guide",
        "doc_year": 2024,
        "region": "indonesia",
        "product_family": "regional",
    },
    {
        "path": DATA_DIR / "DOC-IBG-Vietnam-en-UK-0823_베트남.pdf",
        "doc_type": "benefit_guide",
        "doc_year": 2023,
        "region": "vietnam",
        "product_family": "regional",
    },
    {
        "path": DATA_DIR / "DOC-IBG-HongKong-en-UK-2024_홍콩.pdf",
        "doc_type": "benefit_guide",
        "doc_year": 2023,
        "region": "hong_kong",
        "product_family": "regional",
    },
    {
        "path": DATA_DIR / "DOC-IBG-AZJD-en-UK-0824_중국.pdf",
        "doc_type": "benefit_guide",
        "doc_year": 2024,
        "region": "china",
        "product_family": "regional",
    },
    {
        "path": DATA_DIR / "DOC-SUISSE-IBG-KPT-EN-0624_스위스.pdf",
        "doc_type": "benefit_guide",
        "doc_year": 2022,
        "region": "switzerland",
        "product_family": "regional",
    },
    {
        "path": DATA_DIR / "DOC-IBG-CARE-UK-EN-1125_영국.pdf",
        "doc_type": "benefit_guide",
        "doc_year": 2025,
        "region": "uk",
        "product_family": "regional",
    },
    {
        "path": DATA_DIR / "DOC-IBG-FP-en-UK-1223_프랑스.pdf",
        "doc_type": "benefit_guide",
        "doc_year": 2024,
        "region": "france_benelux_monaco",
        "product_family": "regional",
    },
    {
        "path": DATA_DIR / "DOC-Global-IBG-EN-0524_남미.pdf",
        "doc_type": "benefit_guide",
        "doc_year": 2024,
        "region": "latin_america",
        "product_family": "regional",
    },
]


def clean_text(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = text.replace("\u200b", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def read_pdf_pages(pdf_path: Path) -> List[tuple[int, str]]:
    if not pdf_path.exists():
        print(f"[WARN] 파일이 없습니다: {pdf_path}")
        return []

    pages: List[tuple[int, str]] = []
    doc = fitz.open(pdf_path)

    try:
        for i, page in enumerate(doc):
            text = clean_text(page.get_text("text"))
            if text:
                pages.append((i + 1, text))
    finally:
        doc.close()

    return pages


def build_common_metadata(
    file_info: Dict[str, Any],
    source_name: str,
    page_num: int,
    chunk_idx: int | None = None,
    section: str | None = None,
) -> Dict[str, Any]:
    metadata = {
        "source": source_name,
        "doc_type": file_info["doc_type"],
        "doc_year": file_info["doc_year"],
        "region": file_info["region"],
        "product_family": file_info["product_family"],
        "page": page_num,
        "insurer": "Allianz",
    }

    if chunk_idx is not None:
        metadata["chunk_idx"] = chunk_idx

    if section:
        metadata["section"] = section

    return metadata


REGION_ALIASES = {
    "global": ["global", "worldwide", "전세계", "글로벌", "공통"],
    "singapore": ["singapore", "싱가포르"],
    "dubai_northern_emirates": ["dubai", "northern emirates", "두바이", "북부에미리트", "uae", "아랍에미리트"],
    "lebanon": ["lebanon", "레바논"],
    "indonesia": ["indonesia", "인도네시아"],
    "vietnam": ["vietnam", "베트남"],
    "hong_kong": ["hong kong", "hk", "홍콩"],
    "china": ["china", "중국", "중화권"],
    "switzerland": ["switzerland", "suisse", "스위스"],
    "uk": ["uk", "united kingdom", "england", "britain", "영국"],
    "france_benelux_monaco": ["france", "benelux", "monaco", "프랑스", "모나코", "베네룩스"],
    "latin_america": ["latin america", "남미", "라틴아메리카"],
}

DOC_TYPE_ALIASES = {
    "benefit_guide": [
        "benefit guide", "coverage guide", "benefits", "혜택 가이드", "보장 안내", "보장", "혜택"
    ],
    "tob": [
        "table of benefits", "schedule of benefits", "benefit limits",
        "보장금액", "보장표", "한도표", "한도", "limit"
    ],
    "preauth_form": [
        "pre-authorisation form", "preauthorization form", "preauth form",
        "사전승인 신청서", "사전승인", "입원 전 승인", "직접청구 준비"
    ],
    "claim_form": [
        "claim form", "reimbursement form",
        "보험금 청구서", "청구서", "환급 청구", "사후 청구"
    ],
}

INSURANCE_SEARCH_TAGS = [
    "coverage", "covered", "benefit", "limit", "co-payment", "copay",
    "deductible", "waiting period", "exclusion", "outpatient", "inpatient",
    "maternity", "cancer", "chronic condition", "pre-existing condition",
    "pre-authorisation", "preauthorization", "planned hospitalisation",
    "direct billing", "claim", "reimbursement", "invoice", "receipt",
    "서류", "청구", "환급", "직접청구", "사전승인", "보장", "혜택",
    "한도", "면책", "제외사항", "외래", "입원", "출산", "기왕증"
]


def build_search_tags(file_info: Dict[str, Any]) -> str:
    region_aliases = REGION_ALIASES.get(file_info["region"], [])
    doc_type_aliases = DOC_TYPE_ALIASES.get(file_info["doc_type"], [])

    return "\n".join([
        "[search_tags]",
        f"region: {' | '.join(region_aliases)}",
        f"doc_type: {' | '.join(doc_type_aliases)}",
        f"product_family: {file_info['product_family']}",
        "insurer: Allianz 알리안츠",
        "keywords: " + ", ".join(INSURANCE_SEARCH_TAGS),
    ])


def enrich_text_for_multilingual_search(text: str, file_info: Dict[str, Any]) -> str:
    return f"{text}\n\n{build_search_tags(file_info)}"


def chunk_benefit_guide(
    pages: List[tuple[int, str]],
    source_name: str,
    file_info: Dict[str, Any],
) -> List[Document]:
    docs: List[Document] = []

    for page_num, text in pages:
        paragraphs = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 80]

        for idx, para in enumerate(paragraphs):
            docs.append(
                Document(
                    page_content=enrich_text_for_multilingual_search(para, file_info),
                    metadata=build_common_metadata(
                        file_info=file_info,
                        source_name=source_name,
                        page_num=page_num,
                        chunk_idx=idx,
                    ),
                )
            )
    return docs


def normalize_cell_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = text.replace("\u200b", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

def normalize_benefit_value(text: str) -> str:
    text = clean_text(text)

    if text == "√":
        return "Covered in full"
    if text == "X":
        return "Not covered"

    text = re.sub(r"\b√\b", "Covered in full", text)
    text = re.sub(r"\bX\b", "Not covered", text)

    return text.strip()

def is_noise_line(line: str) -> bool:
    line_l = line.lower().strip()
    noise_patterns = [
        r"^care base care enhanced care signature$",
        r"^core plans key to table of benefits",
        r"^√ covered in full",
        r"^x not available$",
        r"^maximum plan limit$",
        r"^out-patient plans$",
        r"^dental plans$",
        r"^area of cover$",
        r"^click here or press enter",
        r"^looking for something specific\?$",
    ]
    return any(re.search(p, line_l) for p in noise_patterns)


def is_section_header(line: str) -> bool:
    line_l = line.lower().strip()
    section_patterns = [
        r"^in-patient benefits$",
        r"^other benefits$",
        r"^additional core plan services$",
        r"^out-patient plan benefits$",
        r"^dental plan benefits$",
        r"^core plans\b",
        r"^out-patient plans\b",
        r"^dental plans\b",
    ]
    return any(re.search(p, line_l) for p in section_patterns)


def looks_like_row_start(line: str) -> bool:
    s = line.strip()
    s_l = s.lower()

    if not s:
        return False

    banned_fragments = [
        "pre-authorisation required",
        "waiting period applies",
        "in-patient and out-patient treatment",
        "in-patient treatment only",
        "in-patient and day-care treatment only",
        "out-patient treatment only",
        "day-care treatment",
        "per pregnancy",
        "per event",
        "max.",
        "private room",
        "up to",
        "√",
        "x",
        "£",
        "€",
        "us$",
        "chf",
    ]
    if any(x in s_l for x in banned_fragments):
        return False

    if len(s) > 120:
        return False

    return bool(re.match(r"^[A-Z][A-Za-z0-9/\-\(\),&’' ]+$", s))


def build_tob_row_documents_from_page_text(
    text: str,
    page_num: int,
    source_name: str,
    file_info: Dict[str, Any],
) -> List[Document]:
    docs: List[Document] = []
    lines = [normalize_cell_text(line) for line in text.split("\n") if normalize_cell_text(line)]

    current_section = None
    current_row: List[str] = []
    chunk_idx = 0

    def flush_row():
        nonlocal chunk_idx, current_row

        if not current_row:
            return

        row_text = " | ".join(current_row).strip()
        if len(row_text) < 20:
            current_row = []
            return

        structured_text = row_text
        if current_section:
            structured_text = f"Section: {current_section}\n{row_text}"

        docs.append(
            Document(
                page_content=enrich_text_for_multilingual_search(structured_text, file_info),
                metadata=build_common_metadata(
                    file_info=file_info,
                    source_name=source_name,
                    page_num=page_num,
                    chunk_idx=chunk_idx,
                    section=current_section,
                ),
            )
        )
        chunk_idx += 1
        current_row = []

    for line in lines:
        if is_noise_line(line):
            continue

        if is_section_header(line):
            flush_row()
            current_section = line
            continue

        if looks_like_row_start(line):
            flush_row()
            current_row = [line]
            continue

        if current_row:
            current_row.append(line)

    flush_row()
    return docs


def chunk_tob_pdfplumber(
    pdf_path: Path,
    source_name: str,
    file_info: Dict[str, Any],
) -> List[Document]:
    docs: List[Document] = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_idx, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            text = normalize_cell_text(text)
            if not text:
                continue

            docs.extend(
                build_tob_row_documents_from_page_text(
                    text=text,
                    page_num=page_idx,
                    source_name=source_name,
                    file_info=file_info,
                )
            )

    return docs


def chunk_form(
    pages: List[tuple[int, str]],
    source_name: str,
    file_info: Dict[str, Any],
) -> List[Document]:
    docs: List[Document] = []

    for page_num, text in pages:
        blocks = [b.strip() for b in text.split("\n\n") if len(b.strip()) > 40]

        for idx, block in enumerate(blocks):
            docs.append(
                Document(
                    page_content=enrich_text_for_multilingual_search(block, file_info),
                    metadata=build_common_metadata(
                        file_info=file_info,
                        source_name=source_name,
                        page_num=page_num,
                        chunk_idx=idx,
                    ),
                )
            )

    return docs


def build_documents() -> List[Document]:
    all_docs: List[Document] = []

    for file_info in FILES:
        path: Path = file_info["path"]
        source_name = path.name
        doc_type = file_info["doc_type"]

        if not path.exists():
            print(f"[WARN] 파일이 없습니다: {path}")
            continue

        print(
            f"[INFO] 처리 중: {source_name} | type={doc_type} | "
            f"region={file_info['region']} | year={file_info['doc_year']}"
        )
    
        if doc_type == "tob":
            docs = chunk_tob_pdfplumber(path, source_name, file_info)
        else:
            continue
        #     pages = read_pdf_pages(path)
        #     if not pages:
        #         continue

            # if doc_type == "benefit_guide":
            #     docs = chunk_benefit_guide(pages, source_name, file_info)
            # elif doc_type in ["preauth_form", "claim_form"]:
            #     docs = chunk_form(pages, source_name, file_info)
            # else:
            #     print(f"[WARN] 지원하지 않는 doc_type: {doc_type}")
            #     continue

        all_docs.extend(docs)

    return all_docs

In [9]:
documents = build_documents()

for d in documents[:100]:
    # print(d.metadata)
    print(d.page_content)
    print("-" * 50)
# documents

[INFO] 처리 중: DOC-Care-IBG-EN-1125_개인고객용혜택가이드.pdf | type=benefit_guide | region=global | year=2025
[INFO] 처리 중: care-tob-en_보장금액.pdf | type=tob | region=global | year=2025
[INFO] 처리 중: FRM-PreAuth-EN-0825_사전승인신청서.pdf | type=preauth_form | region=global | year=2025
[INFO] 처리 중: FRM-PCF-EN-1125_사후보험청구서.pdf | type=claim_form | region=global | year=2025
[INFO] 처리 중: DOC-Singapore-IBG-EN-0126_싱가포르.pdf | type=benefit_guide | region=singapore | year=2026
[INFO] 처리 중: DOC-IBG-Dubai-Northern-Emirates-EN-0126_두바이.pdf | type=benefit_guide | region=dubai_northern_emirates | year=2026
[INFO] 처리 중: DOC-LEBANON-IBG-EN-0725_레바논.pdf | type=benefit_guide | region=lebanon | year=2025
[INFO] 처리 중: DOC-IBG-Indonesia-en-UK-1123_인도네시아.pdf | type=benefit_guide | region=indonesia | year=2024
[INFO] 처리 중: DOC-IBG-Vietnam-en-UK-0823_베트남.pdf | type=benefit_guide | region=vietnam | year=2023
[INFO] 처리 중: DOC-IBG-HongKong-en-UK-2024_홍콩.pdf | type=benefit_guide | region=hong_kong | year=2023
[INFO] 처리 중: DOC-IBG-AZJD

In [ ]:
print(len(documents))
print(type(documents[0]))
print(documents[0].page_content[:500])
print(documents[0].metadata)

879
<class 'langchain_core.documents.base.Document'>
1/79 
 
 
 
Individual 
Benefit Guide 
 
Care 
International healthcare plans for you 
and your family 
Valid from 1st November 2025

[search_tags]
region: global | worldwide | 전세계 | 글로벌 | 공통
doc_type: benefit guide | coverage guide | benefits | 혜택 가이드 | 보장 안내 | 보장 | 혜택
product_family: care_global
insurer: Allianz 알리안츠
keywords: coverage, covered, benefit, limit, co-payment, copay, deductible, waiting period, exclusion, outpatient, inpatient, maternity, cancer, chronic condition, pre-existing co
{'source': 'DOC-Care-IBG-EN-1125_개인고객용혜택가이드.pdf', 'doc_type': 'benefit_guide', 'doc_year': 2025, 'region': 'global', 'product_family': 'care_global', 'page': 1, 'insurer': 'Allianz', 'chunk_idx': 0}


---

---

# 다른 방식

In [5]:
import re
import json
import math
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

import fitz  # PyMuPDF
import pdfplumber
import pandas as pd

# camelot은 설치 환경에 따라 import 실패 가능
try:
    import camelot
    CAMELOT_AVAILABLE = True
except Exception:
    CAMELOT_AVAILABLE = False


# =========================================================
# 1. 기본 유틸
# =========================================================

def clean_text(text: Optional[str]) -> str:
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\xa0", " ")
    text = text.replace("\u200b", " ")
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_cell(cell: Any) -> str:
    """
    표 셀 텍스트 정리.
    줄바꿈은 완전히 없애지 않고, 의미 있는 경우 '; '로 연결해서 보존.
    """
    text = clean_text(cell)
    if not text:
        return ""

    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]
    if not lines:
        return ""

    # 예: 금액 / max days / waiting period 등 의미 단위는 유지
    # 단, 셀 내부 다중 줄은 한 셀 값으로 합침
    return "; ".join(lines)

def normalize_benefit_value(text: str) -> str:
    text = clean_text(text)

    if not text:
        return ""

    # 단독 기호
    if text == "√":
        return "Covered in full"
    if text == "X":
        return "Not covered"

    # 문장 내부 기호도 치환
    text = text.replace("√", "Covered in full")
    text = re.sub(r"(?<![A-Za-z])X(?![A-Za-z])", "Not covered", text)

    return clean_text(text)

def row_nonempty_ratio(row: List[str]) -> float:
    if not row:
        return 0.0
    nonempty = sum(1 for c in row if clean_text(c))
    return nonempty / len(row)


def is_mostly_empty_row(row: List[str], threshold: float = 0.2) -> bool:
    return row_nonempty_ratio(row) < threshold


def merge_continuation_rows(rows: List[List[str]]) -> List[List[str]]:
    """
    PDF 표 추출 시 설명문/보조문장이 다음 줄로 밀리는 경우가 많음.
    첫 컬럼이 비어 있고 뒤쪽만 채워진 행,
    또는 첫 행 설명의 continuation으로 보이는 행을 이전 행에 병합.
    """
    merged = []

    for row in rows:
        row = [normalize_cell(c) for c in row]

        if not any(row):
            continue

        if not merged:
            merged.append(row)
            continue

        prev = merged[-1]

        # 첫 컬럼 비고 나머지 컬럼만 있는 continuation row
        first_empty = (len(row) > 0 and row[0] == "")
        enough_content = sum(1 for c in row if c) >= 1

        # 이전 행에 이어붙이는 조건
        if first_empty and enough_content:
            max_len = max(len(prev), len(row))
            prev += [""] * (max_len - len(prev))
            row += [""] * (max_len - len(row))

            new_prev = []
            for a, b in zip(prev, row):
                if a and b:
                    new_prev.append(f"{a} | {b}")
                elif b:
                    new_prev.append(b)
                else:
                    new_prev.append(a)
            merged[-1] = new_prev
        else:
            merged.append(row)

    return merged


def forward_fill_header_like_rows(rows: List[List[str]]) -> List[List[str]]:
    """
    일부 PDF 추출 결과에서 상단 헤더가 여러 행으로 분리되므로
    간단한 forward fill 방식으로 보정.
    """
    if not rows:
        return rows

    max_cols = max(len(r) for r in rows)
    norm_rows = [r + [""] * (max_cols - len(r)) for r in rows]

    return norm_rows


def dataframe_from_rows(rows: List[List[str]]) -> pd.DataFrame:
    rows = forward_fill_header_like_rows(rows)
    if not rows:
        return pd.DataFrame()

    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]

    # 첫 1~3행 안에서 헤더 후보 탐색
    header_idx = 0
    best_score = -1

    for i in range(min(3, len(rows))):
        score = row_nonempty_ratio(rows[i])
        if score > best_score:
            best_score = score
            header_idx = i

    header = [normalize_cell(c) if normalize_cell(c) else f"col_{j}" for j, c in enumerate(rows[header_idx])]
    data_rows = rows[header_idx + 1:]

    # 중복 헤더명 방지
    seen = {}
    final_header = []
    for h in header:
        if h not in seen:
            seen[h] = 1
            final_header.append(h)
        else:
            seen[h] += 1
            final_header.append(f"{h}_{seen[h]}")

    df = pd.DataFrame(data_rows, columns=final_header)
    return df


def looks_like_table(df: pd.DataFrame) -> bool:
    """
    추출된 DataFrame이 실제 표인지 대략 판별
    """
    if df.empty:
        return False
    if df.shape[0] < 2 or df.shape[1] < 2:
        return False

    nonempty_cells = (df.fillna("").astype(str).applymap(lambda x: 1 if clean_text(x) else 0).sum().sum())
    total_cells = df.shape[0] * df.shape[1]
    fill_ratio = nonempty_cells / max(total_cells, 1)

    return fill_ratio > 0.25


# =========================================================
# 2. PDF 표 추출
# =========================================================

def extract_tables_with_pdfplumber(pdf_path: str) -> List[Dict[str, Any]]:
    """
    pdfplumber 기반 1차 추출
    """
    results = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_idx, page in enumerate(pdf.pages, start=1):
            page_tables = page.extract_tables(
                table_settings={
                    "vertical_strategy": "lines",
                    "horizontal_strategy": "lines",
                    "intersection_tolerance": 5,
                    "snap_tolerance": 3,
                    "join_tolerance": 3,
                    "edge_min_length": 10,
                    "min_words_vertical": 1,
                    "min_words_horizontal": 1,
                    # "keep_blank_chars": True,
                    "text_x_tolerance": 2,
                    "text_y_tolerance": 2,
                }
            )

            if not page_tables:
                # lines 기반 실패 시 text 기반 재시도
                page_tables = page.extract_tables(
                    table_settings={
                        "vertical_strategy": "text",
                        "horizontal_strategy": "text",
                        "intersection_tolerance": 5,
                        "snap_tolerance": 3,
                        "join_tolerance": 3,
                        "text_x_tolerance": 2,
                        "text_y_tolerance": 2,
                        # "keep_blank_chars": True,
                        "min_words_vertical": 1,
                        "min_words_horizontal": 1,
                    }
                )

            for t_idx, table in enumerate(page_tables, start=1):
                rows = [[normalize_cell(cell) for cell in row] for row in table if row]
                rows = [r for r in rows if any(clean_text(c) for c in r)]
                rows = merge_continuation_rows(rows)
                df = dataframe_from_rows(rows)

                if looks_like_table(df):
                    results.append({
                        "page": page_idx,
                        "extractor": "pdfplumber",
                        "table_index": t_idx,
                        "rows": rows,
                        "df": df
                    })

    return results


def extract_tables_with_camelot(pdf_path: str, pages: str = "all") -> List[Dict[str, Any]]:
    """
    camelot 기반 2차 보강
    """
    if not CAMELOT_AVAILABLE:
        return []

    results = []

    # lattice -> stream 순으로 시도
    for flavor in ["lattice", "stream"]:
        try:
            tables = camelot.read_pdf(
                pdf_path,
                pages=pages,
                flavor=flavor,
                strip_text="\n",
                line_scale=40
            )

            for i, table in enumerate(tables, start=1):
                df = table.df.copy()
                df = df.applymap(normalize_cell)

                # 완전 빈 행 제거
                df = df[~df.apply(lambda r: all(not clean_text(v) for v in r), axis=1)].reset_index(drop=True)

                if looks_like_table(df):
                    results.append({
                        "page": table.page,
                        "extractor": f"camelot_{flavor}",
                        "table_index": i,
                        "rows": df.values.tolist(),
                        "df": df
                    })
        except Exception:
            continue

    return results


def deduplicate_tables(tables: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    동일 페이지에서 비슷한 표가 중복 추출될 수 있어 간단 중복 제거
    """
    seen = set()
    deduped = []

    for t in tables:
        page = t["page"]
        df = t["df"].fillna("").astype(str)
        signature = (page, df.shape, "|".join(df.head(5).astype(str).agg("||".join, axis=1).tolist()))

        if signature not in seen:
            seen.add(signature)
            deduped.append(t)

    deduped.sort(key=lambda x: (x["page"], x["table_index"]))
    return deduped


# =========================================================
# 3. 섹션/플랜/행 구조화
# =========================================================

SECTION_PATTERNS = [
    ("core_plans", re.compile(r"\bcore plans\b", re.I)),
    ("outpatient_plans", re.compile(r"\bout-?patient plans\b", re.I)),
    ("dental_plans", re.compile(r"\bdental plans\b", re.I)),
    ("area_of_cover", re.compile(r"\barea of cover\b", re.I)),
    ("deductibles", re.compile(r"\bdeductibles?\b", re.I)),
]

PLAN_NAMES = ["Care Base", "Care Enhanced", "Care Signature"]


def detect_section_from_page_text(page_text: str) -> str:
    txt = clean_text(page_text)
    for name, pattern in SECTION_PATTERNS:
        if pattern.search(txt):
            return name
    return "unknown"


def extract_page_texts(pdf_path: str) -> Dict[int, str]:
    doc = fitz.open(pdf_path)
    page_texts = {}
    for i, page in enumerate(doc, start=1):
        page_texts[i] = clean_text(page.get_text("text"))
    doc.close()
    return page_texts


def find_plan_columns(df: pd.DataFrame) -> Dict[str, Optional[str]]:
    """
    컬럼명에서 Care Base / Care Enhanced / Care Signature에 해당하는 컬럼을 찾음
    """
    mapping = {p: None for p in PLAN_NAMES}

    for col in df.columns:
        col_clean = clean_text(col).lower()
        if "care base" in col_clean:
            mapping["Care Base"] = col
        elif "care enhanced" in col_clean:
            mapping["Care Enhanced"] = col
        elif "care signature" in col_clean:
            mapping["Care Signature"] = col

    # 헤더가 깨진 경우 위치 기반 fallback
    if not any(mapping.values()) and df.shape[1] >= 4:
        cols = list(df.columns)
        mapping["Care Base"] = cols[-3]
        mapping["Care Enhanced"] = cols[-2]
        mapping["Care Signature"] = cols[-1]

    return mapping


def infer_benefit_column(df: pd.DataFrame, plan_cols: Dict[str, Optional[str]]) -> Optional[str]:
    plan_col_names = {v for v in plan_cols.values() if v}
    for col in df.columns:
        if col not in plan_col_names:
            return col
    return df.columns[0] if len(df.columns) > 0 else None


def dataframe_to_structured_rows(df: pd.DataFrame, page: int, section: str) -> List[Dict[str, Any]]:
    valid_columns = [c for c in df.columns if not is_noise_column(c)]
    df = df[valid_columns].copy()
    
    plan_cols = find_plan_columns(df)
    benefit_col = infer_benefit_column(df, plan_cols)

    rows_structured = []

    for idx, row in df.iterrows():
        benefit_name = clean_text(row.get(benefit_col, "")) if benefit_col else ""

        # 너무 빈약한 행 제거
        row_values = {c: normalize_benefit_value(row.get(c, "")) for c in df.columns}
        if not any(row_values.values()):
            continue

        rows_structured.append({
            "page": page,
            "section": section,
            "row_index_in_table": idx,
            "benefit": benefit_name,
            "raw_row": row_values,
            "plans": {
                "Care Base": normalize_benefit_value(row.get(plan_cols["Care Base"], "")) if plan_cols["Care Base"] else "",
                "Care Enhanced": normalize_benefit_value(row.get(plan_cols["Care Enhanced"], "")) if plan_cols["Care Enhanced"] else "",
                "Care Signature": normalize_benefit_value(row.get(plan_cols["Care Signature"], "")) if plan_cols["Care Signature"] else "",
            }
        })

    return rows_structured

def is_noise_column(col_name: str) -> bool:
    col = clean_text(col_name)
    if not col:
        return True

    noise_patterns = [
        r"^√(_\d+)?$",
        r"^X(_\d+)?$",
        r"^covered in full(_\d+)?$",
        r"^not covered(_\d+)?$"
    ]

    for p in noise_patterns:
        if re.match(p, col, flags=re.I):
            return True

    return False

# =========================================================
# 4. 표 -> RAG 청크 생성
# =========================================================
def structured_row_to_text(row: Dict[str, Any]) -> str:
    benefit = normalize_benefit_value(row.get("benefit", ""))
    plans = row.get("plans", {})

    parts = []
    if benefit:
        parts.append(f"Benefit: {benefit}")

    for plan_name in PLAN_NAMES:
        val = normalize_benefit_value(plans.get(plan_name, ""))
        if val:
            parts.append(f"{plan_name}: {val}")

    raw_row = row.get("raw_row", {})
    extra_parts = []
    for k, v in raw_row.items():
        k = clean_text(k)
        v = normalize_benefit_value(v)

        if not v:
            continue
        if k in PLAN_NAMES:
            continue
        if "care " in k.lower():
            continue
        if is_noise_column(k):
            continue

        extra_parts.append(f"{k}: {v}")

    if extra_parts:
        parts.append(" | ".join(extra_parts))

    return "\n".join(parts).strip()


def chunk_rows(
    rows: List[Dict[str, Any]],
    max_chars: int = 2500,
    min_rows_per_chunk: int = 2,
    overlap_rows: int = 1
) -> List[List[Dict[str, Any]]]:
    """
    표 행 단위 청킹.
    한 행이라도 분리하지 않고 행 묶음으로 청킹.
    """
    if not rows:
        return []

    chunks = []
    current = []
    current_len = 0

    for row in rows:
        row_text = structured_row_to_text(row)
        row_len = len(row_text)

        if current and current_len + row_len > max_chars and len(current) >= min_rows_per_chunk:
            chunks.append(current)

            if overlap_rows > 0:
                current = current[-overlap_rows:]
                current_len = sum(len(structured_row_to_text(r)) for r in current)
            else:
                current = []
                current_len = 0

        current.append(row)
        current_len += row_len

    if current:
        chunks.append(current)

    return chunks


def build_chunk_text(chunk_rows_data: List[Dict[str, Any]], table_meta: Dict[str, Any]) -> str:
    page = table_meta["page"]
    section = table_meta["section"]
    extractor = table_meta["extractor"]

    lines = [
        f"[SECTION] {section}",
        f"[PAGE] {page}",
        f"[EXTRACTOR] {extractor}",
        f"[ROW_COUNT] {len(chunk_rows_data)}",
        ""
    ]

    for row in chunk_rows_data:
        lines.append(structured_row_to_text(row))
        lines.append("")

    return "\n".join(lines).strip()


def build_chunk_record(
    chunk_id: str,
    chunk_rows_data: List[Dict[str, Any]],
    table_meta: Dict[str, Any]
) -> Dict[str, Any]:
    text = build_chunk_text(chunk_rows_data, table_meta)

    return {
        "chunk_id": chunk_id,
        "doc_id": table_meta["doc_id"],
        "source_file": table_meta["source_file"],
        "page_start": min(r["page"] for r in chunk_rows_data),
        "page_end": max(r["page"] for r in chunk_rows_data),
        "section": table_meta["section"],
        "table_index": table_meta["table_index"],
        "extractor": table_meta["extractor"],
        "row_count": len(chunk_rows_data),
        "benefit_names": [r.get("benefit", "") for r in chunk_rows_data if r.get("benefit", "")],
        "text": text,
        "rows_structured": chunk_rows_data,
        "raw_table_text": "\n\n".join(structured_row_to_text(r) for r in chunk_rows_data)
    }


# =========================================================
# 5. 메인 파이프라인
# =========================================================

def extract_and_chunk_tables_for_rag(
    pdf_path: str,
    output_dir: str = "./rag_chunks",
    max_chars: int = 2500
) -> List[Dict[str, Any]]:
    pdf_path = str(pdf_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    source_file = Path(pdf_path).name
    doc_id = Path(pdf_path).stem

    page_texts = extract_page_texts(pdf_path)

    # 1차: pdfplumber
    plumber_tables = extract_tables_with_pdfplumber(pdf_path)

    # 2차: camelot 보강
    camelot_tables = extract_tables_with_camelot(pdf_path)

    # 병합 및 중복 제거
    all_tables = deduplicate_tables(plumber_tables + camelot_tables)

    if not all_tables:
        raise RuntimeError("표를 추출하지 못했습니다. PDF 구조를 다시 확인해야 합니다.")

    chunk_records = []
    chunk_seq = 1

    for table in all_tables:
        page = table["page"]
        section = detect_section_from_page_text(page_texts.get(page, ""))
        df = table["df"].copy()

        # 컬럼명이 숫자 인덱스뿐이면 첫 행을 헤더 후보로 한번 더 승격
        if all(str(c).isdigit() or str(c).startswith("col_") for c in df.columns):
            rows = df.values.tolist()
            df = dataframe_from_rows(rows)

        rows_structured = dataframe_to_structured_rows(df, page=page, section=section)

        # 너무 빈약하면 skip
        if len(rows_structured) < 1:
            continue

        table_meta = {
            "doc_id": doc_id,
            "source_file": source_file,
            "page": page,
            "section": section,
            "table_index": table["table_index"],
            "extractor": table["extractor"],
        }

        row_chunks = chunk_rows(
            rows_structured,
            max_chars=max_chars,
            min_rows_per_chunk=2,
            overlap_rows=1
        )

        for rc in row_chunks:
            chunk_id = f"{doc_id}_p{page}_t{table['table_index']}_c{chunk_seq:04d}"
            rec = build_chunk_record(chunk_id, rc, table_meta)
            chunk_records.append(rec)
            chunk_seq += 1

    # 저장
    jsonl_path = output_dir / f"{doc_id}_table_chunks.jsonl"
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for rec in chunk_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    # 디버그용 CSV도 저장
    summary_rows = []
    for rec in chunk_records:
        summary_rows.append({
            "chunk_id": rec["chunk_id"],
            "section": rec["section"],
            "page_start": rec["page_start"],
            "page_end": rec["page_end"],
            "row_count": rec["row_count"],
            "benefit_names": " | ".join(rec["benefit_names"][:5]),
            "text_preview": rec["text"][:300]
        })
    pd.DataFrame(summary_rows).to_csv(output_dir / f"{doc_id}_chunk_summary.csv", index=False, encoding="utf-8-sig")

    return chunk_records


# =========================================================
# 6. 검색 테스트용 간단 함수
# =========================================================

def load_chunks(jsonl_path: str) -> List[Dict[str, Any]]:
    records = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records


def simple_keyword_search(chunks: List[Dict[str, Any]], query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    q = clean_text(query).lower()
    scored = []

    for ch in chunks:
        text = clean_text(ch["text"]).lower()
        score = 0

        # 단순 점수: query 단어 포함 수
        for token in q.split():
            if token in text:
                score += text.count(token)

        if score > 0:
            scored.append((score, ch))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [item[1] for item in scored[:top_k]]


# =========================================================
# 7. 실행 예시
# =========================================================
if __name__ == "__main__":
    pdf_path = r"C:\skn24\allianz-rag\data\raw\care-tob-en_보장금액.pdf"
    output_dir = r"./rag_chunks"

    chunks = extract_and_chunk_tables_for_rag(
        pdf_path=pdf_path,
        output_dir=output_dir,
        max_chars=2500
    )

    print(f"총 청크 수: {len(chunks)}")
    print("저장 완료:")
    print(f"- JSONL: {Path(output_dir) / (Path(pdf_path).stem + '_table_chunks.jsonl')}")
    print(f"- CSV  : {Path(output_dir) / (Path(pdf_path).stem + '_chunk_summary.csv')}")

    # 검색 예시
    query = "Out-patient surgery pre-authorisation nursing at home max 90 days"
    results = simple_keyword_search(chunks, query, top_k=3)

    print("\n[검색 결과]")
    for r in results:
        print("=" * 80)
        print(r["chunk_id"], f"(page {r['page_start']}~{r['page_end']}, section={r['section']})")
        print(r["text"][:1200])

C:\Users\Playdata\AppData\Local\Temp\ipykernel_23896\1251620398.py:185: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  nonempty_cells = (df.fillna("").astype(str).applymap(lambda x: 1 if clean_text(x) else 0).sum().sum())
C:\Users\Playdata\AppData\Local\Temp\ipykernel_23896\1251620398.py:185: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  nonempty_cells = (df.fillna("").astype(str).applymap(lambda x: 1 if clean_text(x) else 0).sum().sum())
C:\Users\Playdata\AppData\Local\Temp\ipykernel_23896\1251620398.py:185: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  nonempty_cells = (df.fillna("").astype(str).applymap(lambda x: 1 if clean_text(x) else 0).sum().sum())
C:\Users\Playdata\AppData\Local\Temp\ipykernel_23896\1251620398.py:185: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  nonempty_cells = (df.fillna("").astype(str).applymap(lambda x: 1 if

총 청크 수: 35
저장 완료:
- JSONL: rag_chunks\care-tob-en_보장금액_table_chunks.jsonl
- CSV  : rag_chunks\care-tob-en_보장금액_chunk_summary.csv

[검색 결과]
care-tob-en_보장금액_p10_t8_c0015 (page 10~10, section=core_plans)
[SECTION] core_plans
[PAGE] 10
[EXTRACTOR] camelot_lattice
[ROW_COUNT] 7

Benefit: Travel costs of insured family members in the event of anPre-authorisationevacuation/repatriationrequired
Travel costs for one person accompanying an evacuated/Pre-authorisationrepatriated personrequired: Travel costs of insured family members in the event of anPre-authorisationevacuation/repatriationrequired | £ 1,660 / € 2,000 /US$ 2,700 / CHF 2,600: Not covered

Benefit: Travel costs of insured members to be with a close relative who isat peril of death or who has died
Travel costs for one person accompanying an evacuated/Pre-authorisationrepatriated personrequired: Travel costs of insured members to be with a close relative who isat peril of death or who has died | £ 1,660 / € 2,000 /US$ 2,700 / CHF 2,6

C:\Users\Playdata\AppData\Local\Temp\ipykernel_23896\1251620398.py:277: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(normalize_cell)
C:\Users\Playdata\AppData\Local\Temp\ipykernel_23896\1251620398.py:185: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  nonempty_cells = (df.fillna("").astype(str).applymap(lambda x: 1 if clean_text(x) else 0).sum().sum())
C:\Users\Playdata\AppData\Local\Temp\ipykernel_23896\1251620398.py:277: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(normalize_cell)
C:\Users\Playdata\AppData\Local\Temp\ipykernel_23896\1251620398.py:185: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  nonempty_cells = (df.fillna("").astype(str).applymap(lambda x: 1 if clean_text(x) else 0).sum().sum())
C:\Users\Playdata\AppData\Local\Temp\ipykernel_23896\1251620398.py:277: FutureWarning: DataFrame.applyma

In [6]:
import re
import json
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any, Optional
import pdfplumber
import fitz

# camelot 설치 여부 체크
try:
    import camelot
    CAMELOT_AVAILABLE = True
except Exception:
    CAMELOT_AVAILABLE = False

# =========================================================
# 1. 기본 유틸리티 및 전처리
# =========================================================

def clean_text(text: Optional[str]) -> str:
    if text is None: return ""
    text = str(text).replace("\xa0", " ").replace("\u200b", " ").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def normalize_cell(cell: Any) -> str:
    text = clean_text(cell)
    if not text: return ""
    return " ".join([ln.strip() for ln in text.split("\n") if ln.strip()])

def row_nonempty_ratio(row: List[str]) -> float:
    if not row: return 0.0
    return sum(1 for c in row if clean_text(c)) / len(row)

# =========================================================
# 2. PDF 표 추출 (개선 버전)
# =========================================================

def extract_tables_with_pdfplumber(pdf_path: str) -> List[Dict[str, Any]]:
    results = []
    # 텍스트 결합도를 높이기 위한 설정값 조정
    TABLE_SETTINGS = {
        "vertical_strategy": "lines",
        "horizontal_strategy": "lines",
        "intersection_tolerance": 15,
        "snap_tolerance": 6,
        "join_tolerance": 6,
        "edge_min_length": 15,
    }

    with pdfplumber.open(pdf_path) as pdf:
        for page_idx, page in enumerate(pdf.pages, start=1):
            page_tables = page.extract_tables(table_settings=TABLE_SETTINGS)

            # 선(lines) 기반 실패 시 텍스트 간격 기반 재시도
            if not page_tables:
                page_tables = page.extract_tables(table_settings={
                    "vertical_strategy": "text",
                    "horizontal_strategy": "text",
                    "snap_tolerance": 8,
                })

            for t_idx, table in enumerate(page_tables, start=1):
                if not table: continue
                
                # 데이터 정제
                rows = [[normalize_cell(cell) for cell in row] for row in table]
                rows = [r for r in rows if any(c for c in r)] # 완전 빈 행 제거
                
                # DataFrame 변환 및 검증
                if len(rows) < 2: continue
                df = pd.DataFrame(rows[1:], columns=[c if c else f"col_{i}" for i, c in enumerate(rows[0])])
                
                results.append({
                    "page": page_idx,
                    "extractor": "pdfplumber",
                    "table_index": t_idx,
                    "df": df
                })
    return results

# =========================================================
# 3. 구조화 및 청킹 (LLM 최적화)
# =========================================================

def build_chunk_text(df: pd.DataFrame, meta: Dict[str, Any]) -> str:
    """
    LLM이 가장 선호하는 Markdown Table 형식으로 텍스트 구성
    """
    md_table = df.to_markdown(index=False)
    header = f"[SECTION] {meta.get('section', 'unknown')}\n[PAGE] {meta['page']}\n"
    return f"{header}\n{md_table}"

def process_pdf_to_rag_jsonl(pdf_path: str, output_path: str):
    print(f"🔄 처리 시작: {pdf_path}")
    
    # 1. 표 추출
    tables = extract_tables_with_pdfplumber(pdf_path)
    
    # 2. 섹션 인식을 위한 전체 텍스트 추출
    doc = fitz.open(pdf_path)
    page_texts = {i: page.get_text() for i, page in enumerate(doc, start=1)}
    
    chunk_records = []
    for t in tables:
        page_num = t["page"]
        df = t["df"]
        
        # 간단한 섹션 키워드 매칭
        section = "Unknown"
        text_context = page_texts.get(page_num, "").lower()
        if "core plan" in text_context: section = "Core Plans"
        elif "out-patient" in text_context: section = "Out-patient Plans"
        elif "benefit" in text_context: section = "Table of Benefits"

        meta = {"page": page_num, "section": section, "extractor": t["extractor"]}
        
        # 청크 생성
        chunk_text = build_chunk_text(df, meta)
        
        record = {
            "chunk_id": f"{Path(pdf_path).stem}_p{page_num}_t{t['table_index']}",
            "text": chunk_text,
            "metadata": meta
        }
        chunk_records.append(record)

    # 3. 저장
    with open(output_path, "w", encoding="utf-8") as f:
        for rec in chunk_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            
    print(f"✅ 저장 완료: {output_path} (총 {len(chunk_records)} 청크)")

# 실행
if __name__ == "__main__":
    target_pdf = r"C:\skn24\allianz-rag\data\raw\care-tob-en_보장금액.pdf"
    output_jsonl = "./rag_chunks_v2.jsonl"
    process_pdf_to_rag_jsonl(target_pdf, output_jsonl)

🔄 처리 시작: C:\skn24\allianz-rag\data\raw\care-tob-en_보장금액.pdf
✅ 저장 완료: ./rag_chunks_v2.jsonl (총 17 청크)
